In [31]:
import pandas as pd
import numpy as np
from decider.infer import Decider
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, precision_recall_curve, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [2]:
data = pd.read_csv('../data/MACCRs.tsv', sep='\t')
data = data.fillna("Data not available")
print(data.shape)
data.head()

(3100, 59)


,Unnamed: 0,CR Number,Access Date,Total Score,Identification Score,Title,Authors,Year,Journal,Institution,...,Otorhinolaryngologic Diseases,Skin Diseases,Rare Diseases,AllDiseaseSystems,Images,Graphs or Illustrations,Videos,Tables,Contributor,Congenital Heart Disease
0,0,CCR611,Jun 27 2017,23.0,10.0,notalgia paresthetica and multiple endocrine n...,Alcántara F;Feito M;Albizuri F;Beato M;De Lucas R,2016,pediatric dermatology,"department of dermatology, hospital universita...",...,0,1,0,endocrine; skin,2,0,0,0,JZ,0
1,1,CCR1087,Aug 18 2017,20.0,10.0,early cecal cancer adjacent to the appendiceal...,Iwai T;Hotta K;Imai K;Yamaguchi Y;Ito S;Kawata...,2015,gastrointestinal endoscopy,"division of endoscopy, shizuoka cancer center,...",...,0,0,0,cancer; digestive,4,0,1,0,TC,0
2,2,CCR190,Jan 06 2017,23.0,10.0,autologous stem cell transplantation for stiff...,Sanders S;Bredeson C;Pringle CE;Martin L;Allan...,2014,jama neurology,"department of medicine, university of ottawa, ...",...,0,0,0,nervous,2,0,0,0,JZ,0
3,3,CCR1134,Feb 13 2018,24.0,9.0,bone metastasis as the first sign of gastric c...,Ameur WB;Belghali S;Akkari I;Zaghouani H;Bouaj...,2017,pan african medical journal,department of internal medicine and infectious...,...,0,0,0,cancer; digestive; musculoskeletal and rheumat...,1,0,0,0,HC,0
4,4,CCR2978,Oct 26 2017,23.0,8.0,value of radionuclide assessment with thallium...,Taillard F;Mundler O;Tillous-Borde I;Desbois J...,1988,european society of cardiology,"laboratoire de neurochimie, inserm u 134, hopi...",...,0,0,0,cardiovascular,0,2,0,3,JP,0


In [3]:
data.columns

Index(['Unnamed: 0', 'CR Number', 'Access Date', 'Total Score',
       'Identification Score', 'Title', 'Authors', 'Year', 'Journal',
       'Institution', 'PMID', 'DOI', 'Link', 'Medical Content Score',
       'Keywords', 'Geographic Locations', 'Life Style', 'Family History',
       'Social History', 'Medical/Surgical History', 'Signs and Symptoms',
       'Comorbidities', 'Diagnostic Techniques and Procedures', 'Diagnosis',
       'Laboratory Values', 'Pathology', 'Pharmacological Therapy',
       'Interventional Therapy', 'Patient Outcome Assessment',
       'Acknowledgements Score', 'Funding Source', 'Award Number',
       'Disclosures', 'References', 'Age', 'Gender', 'Cancer',
       'Nervous System Diseases', 'Cardiovascular Diseases',
       'Musculoskeletal Diseases and Rheumatological Diseases',
       'Digestive System Diseases', 'Obstetrical and Gynecological Diseases',
       'Infectious Diseases', 'Respiratory Tract Diseases',
       'Hematologic Diseases', 'Kidney Diseas

In [4]:
X = data[['Age', 'Gender', 'Life Style', 'Family History',
       'Social History', 'Medical/Surgical History', 'Signs and Symptoms',
       'Comorbidities', 'Diagnostic Techniques and Procedures','Pathology']]

y = data[['Cardiovascular Diseases']]


print(X.shape)
print(y.shape)

(3100, 10)
(3100, 1)


In [5]:
X.head()

,Age,Gender,Life Style,Family History,Social History,Medical/Surgical History,Signs and Symptoms,Comorbidities,Diagnostic Techniques and Procedures,Pathology
0,9.0,female,Data not available,mother had had medullary thyroid carcinoma at ...,Data not available,a prophylactic thyroidectomy when was 3 years old,itching in the right shoulder region that had ...,Data not available,biopsy,skin biopsy showed perivascular melanophages i...
1,75.0,male,Data not available,Data not available,Data not available,Data not available,colon cancer,Data not available,colonoscopy; magnified narrow-band imaging; ma...,pathologic diagnosis confirmed that the slight...
2,48.0,female,Data not available,Data not available,Data not available,Data not available,"progressive leg stiffness, hyperreflexia, and ...",Data not available,findings on magnetic resonance imaging of her ...,Data not available
3,60.0,female,Data not available,family history was negative for any malignancies,Data not available,history of rheumatoid arthritis; currently in ...,"since 2 months, she describes inflammatory low...",rheumatoid arthritis,x ray of the spine; magnetic resonance imaging...,Data not available
4,2.0,male,Data not available,"the patient, a boy, was the only child of cons...",Data not available,anorexia and vomiting occurred from 7 to 17 mo...,"the patient, a boy, was the only child of cons...",Data not available,growth velocity was normal at ± 1 sd for heigh...,the case of a 30 month-old boy who presented w...


In [6]:
y['Cardiovascular Diseases'].value_counts()

Cardiovascular Diseases
0    2133
1     967
Name: count, dtype: int64

In [7]:
X.columns

Index(['Age', 'Gender', 'Life Style', 'Family History', 'Social History',
       'Medical/Surgical History', 'Signs and Symptoms', 'Comorbidities',
       'Diagnostic Techniques and Procedures', 'Pathology'],
      dtype='str')

In [8]:
def profile_creator(row):
    return f"""Age: {row['Age']}
    Gender: {row['Gender']}
    Life Style: {row['Life Style']}
    Family History: {row['Family History']}
    Social History: {row['Social History']}
    Medical/Surgical History: {row['Medical/Surgical History']}
    Signs and Symptoms: {row['Signs and Symptoms']}
    Comorbidities: {row['Comorbidities']}
    Diagnostic Techniques and Procedures: {row['Diagnostic Techniques and Procedures']}
    Pathology: {row['Pathology']}"""

In [9]:
X['Patient_profile'] = X.apply(lambda x: profile_creator(x), axis=1)
print(X.shape)
X.head()

(3100, 11)


,Age,Gender,Life Style,Family History,Social History,Medical/Surgical History,Signs and Symptoms,Comorbidities,Diagnostic Techniques and Procedures,Pathology,Patient_profile
0,9.0,female,Data not available,mother had had medullary thyroid carcinoma at ...,Data not available,a prophylactic thyroidectomy when was 3 years old,itching in the right shoulder region that had ...,Data not available,biopsy,skin biopsy showed perivascular melanophages i...,Age: 9.0\n Gender: female\n Life Style: ...
1,75.0,male,Data not available,Data not available,Data not available,Data not available,colon cancer,Data not available,colonoscopy; magnified narrow-band imaging; ma...,pathologic diagnosis confirmed that the slight...,Age: 75.0\n Gender: male\n Life Style: D...
2,48.0,female,Data not available,Data not available,Data not available,Data not available,"progressive leg stiffness, hyperreflexia, and ...",Data not available,findings on magnetic resonance imaging of her ...,Data not available,Age: 48.0\n Gender: female\n Life Style:...
3,60.0,female,Data not available,family history was negative for any malignancies,Data not available,history of rheumatoid arthritis; currently in ...,"since 2 months, she describes inflammatory low...",rheumatoid arthritis,x ray of the spine; magnetic resonance imaging...,Data not available,Age: 60.0\n Gender: female\n Life Style:...
4,2.0,male,Data not available,"the patient, a boy, was the only child of cons...",Data not available,anorexia and vomiting occurred from 7 to 17 mo...,"the patient, a boy, was the only child of cons...",Data not available,growth velocity was normal at ± 1 sd for heigh...,the case of a 30 month-old boy who presented w...,Age: 2.0\n Gender: male\n Life Style: Da...


In [10]:
X.at[0, 'Patient_profile']

'Age: 9.0\n    Gender: female\n    Life Style: Data not available\n    Family History: mother had had medullary thyroid carcinoma at age 37 years, pheochromocytoma at 39 years, and had died at age 43 years; the father was healthy\n    Social History: Data not available\n    Medical/Surgical History: a prophylactic thyroidectomy when was 3 years old\n    Signs and Symptoms: itching in the right shoulder region that had begun 2 years before; appearance and growth of a hyperpigmented lesion\n    Comorbidities: Data not available\n    Diagnostic Techniques and Procedures: biopsy\n    Pathology: skin biopsy showed perivascular melanophages in papillary dermis, with no further findings'

In [11]:
d = Decider("Mapika/decider-0.8b")

Loading weights: 100%|██████████| 320/320 [00:01<00:00, 222.45it/s]


In [12]:
question = {
    "cardiovascular": {
    "question": "Does this patient case describe a cardiovascular disease?",
    "type": "noul"}
}

In [ ]:
# Mini test
patient = X["Patient_profile"].iloc[0]
res = y['Cardiovascular Diseases'].iloc[0]

print(patient)
print(res)

Age: 9.0
    Gender: female
    Life Style: Data not available
    Family History: mother had had medullary thyroid carcinoma at age 37 years, pheochromocytoma at 39 years, and had died at age 43 years; the father was healthy
    Social History: Data not available
    Medical/Surgical History: a prophylactic thyroidectomy when was 3 years old
    Signs and Symptoms: itching in the right shoulder region that had begun 2 years before; appearance and growth of a hyperpigmented lesion
    Comorbidities: Data not available
    Diagnostic Techniques and Procedures: biopsy
    Pathology: skin biopsy showed perivascular melanophages in papillary dermis, with no further findings
0


In [30]:
result = d.system_one(
    state=patient,
    questions=question
)

print(result)

{'model': 'decider-0.8b-v1', 'answers': {'cardiovascular': {'type': 'noul', 'noul': 0.1497}}, 'usage': {'input_tokens': 193, 'output_tokens': 0}}


### Test set

In [34]:
positives = X[y['Cardiovascular Diseases']==1]
print(positives.shape)

negatives = X[y['Cardiovascular Diseases']==0]
print(negatives.shape)

(967, 11)
(2133, 11)


In [43]:
Xsample = pd.concat([
    positives.sample(5, random_state=12),
    negatives.sample(5, random_state=12)
])
print(Xsample.shape)
Xsample = Xsample.sort_values(by=['Signs and Symptoms'])
Xsample.head()

(10, 11)


,Age,Gender,Life Style,Family History,Social History,Medical/Surgical History,Signs and Symptoms,Comorbidities,Diagnostic Techniques and Procedures,Pathology,Patient_profile
1387,72.0,female,Data not available,Data not available,Data not available,"arterial hypertension, diabetes mellitus, perm...",Data not available,arterial hypertension; diabetes mellitus; perm...,an echocardiographic examination showed dilate...,Data not available,Age: 72.0\n Gender: female\n Life Style:...
2240,0.0,female,Data not available,Data not available,Data not available,a 10-month-old girl presented with a two-month...,a 10-month-old girl presented with a two-month...,na,examination showed a displaced apex beat and g...,na,Age: 0.0\n Gender: female\n Life Style: ...
555,5.0,female,Data not available,Data not available,Data not available,Data not available,acute abdominal pain,Data not available,computer tomography (ct) scan of the abdo-men ...,cut surfaces of the masswere partly solid with...,Age: 5.0\n Gender: female\n Life Style: ...
933,0.0,male,Data not available,a 19-year-old primigravida mother,Data not available,born via vaginal delivery,"at 9 weeks of life, he experienced further det...",Data not available,postnatal echocardiogram confirmed hlhs with a...,Data not available,Age: 0.0\n Gender: male\n Life Style: Da...
2445,3.0,female,Data not available,the patient's identical twin sister had died u...,Data not available,Data not available,"consanguinity or, occasionally, low plasma car...",Data not available,the blood ammonia level was increased; a muscl...,Data not available,Age: 3.0\n Gender: female\n Life Style: ...


In [44]:
ysample = y.loc[Xsample.index]
print(ysample.shape)
ysample.head(10)

(10, 1)


,Cardiovascular Diseases
1387,1
2240,1
555,0
933,1
2445,1
13,1
328,0
616,0
2532,0
1179,0


In [48]:
results = []
for idx, row in Xsample.iterrows():
    result = d.system_one(
    state=row['Patient_profile'],
    questions=question
    )
    
    results.append({
        "index":idx,
        "y_true":y.loc[idx, "Cardiovascular Diseases"],
        "y_prob": result['answers']['cardiovascular']['noul']
    })


In [49]:
results_df = pd.DataFrame(results)
print(results_df.shape)
results_df.head(10)

(10, 3)


,index,y_true,y_prob
0,1387,1,0.9771
1,2240,1,0.8474
2,555,0,0.0091
3,933,1,0.8754
4,2445,1,0.6437
5,13,1,0.8880
6,328,0,0.7328
7,616,0,0.0206
8,2532,0,0.0391
9,1179,0,0.0715


In [50]:
results_df.groupby("y_true")["y_prob"].describe()

,count,mean,std,min,25%,50%,75%,max
y_true,,,,,,,,
0,5.0,0.17462,0.312923,0.0091,0.0206,0.0391,0.0715,0.7328
1,5.0,0.84632,0.123230,0.6437,0.8474,0.8754,0.8880,0.9771


### Run on full data

In [13]:
results = []

for idx, row in tqdm(X.iterrows(), total=len(X), desc="Running inference"):
    result = d.system_one(
    state=row['Patient_profile'],
    questions=question
    )
    
    results.append({
        "index":idx,
        "y_true":y.loc[idx, "Cardiovascular Diseases"],
        "y_prob": result['answers']['cardiovascular']['noul']
    })


Running inference: 100%|██████████| 3100/3100 [20:48<00:00,  2.48it/s]


In [15]:
results_df = pd.DataFrame(results)
print(results_df.shape)
results_df.head(10)

(3100, 3)


,index,y_true,y_prob
0,0,0,0.1497
1,1,0,0.0040
2,2,0,0.0305
3,3,0,0.0179
4,4,1,0.9514
5,5,0,0.0312
6,6,1,0.9544
7,7,0,0.0048
8,8,1,0.7161
9,9,0,0.0108


In [16]:
results_df.groupby("y_true")["y_prob"].describe()

,count,mean,std,min,25%,50%,75%,max
y_true,,,,,,,,
0,2133.0,0.166450,0.238757,0.0008,0.0126,0.0446,0.2083,0.9594
1,967.0,0.777996,0.253878,0.0038,0.7438,0.8811,0.9434,0.9887


In [17]:
results_df.to_csv("../output/mccr_raw_probabilities.tsv", sep='\t', index=False, encoding='utf-8-sig')

In [20]:
pr_auc = average_precision_score(
    results_df["y_true"],
    results_df["y_prob"]
)

print(f"PR-AUC: {pr_auc:.4f}")

PR-AUC: 0.8896


In [23]:
precision, recall, thresholds = precision_recall_curve(
    results_df["y_true"],
    results_df["y_prob"]
)

pr_curve = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1]
})

pr_curve.head()

,threshold,precision,recall
0,0.0008,0.311935,1.0
1,0.0009,0.312036,1.0
2,0.0010,0.312238,1.0
3,0.0011,0.312339,1.0
4,0.0012,0.312439,1.0


In [ ]:
# For a threshold of 0.5 (exploratory threshold)
y_true = results_df["y_true"]
y_pred = (results_df["y_prob"] >= 0.5).astype(int)

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
print(f"F1:        {f1_score(y_true, y_pred):.4f}")

confusion_matrix(y_true, y_pred)

Accuracy:  0.8710
Precision: 0.7570
Recall:    0.8635
F1:        0.8068


array([[1865,  268],
       [ 132,  835]])

### FP and FN examples

In [29]:
false_positives = results_df[
    (results_df["y_true"] == 0) &
    (results_df["y_prob"] >= 0.5)
].sort_values("y_prob", ascending=False)

false_positives.head(10)

,index,y_true,y_prob
2421,2421,0,0.9594
725,725,0,0.9582
232,232,0,0.9538
2827,2827,0,0.9470
977,977,0,0.9442
2207,2207,0,0.9434
1009,1009,0,0.9434
2380,2380,0,0.9414
2269,2269,0,0.9397
2320,2320,0,0.9329


In [30]:
false_negatives = results_df[
    (results_df["y_true"] == 1) &
    (results_df["y_prob"] < 0.5)
].sort_values("y_prob")

false_negatives.head(10)

,index,y_true,y_prob
3013,3013,1,0.0038
2404,2404,1,0.0087
3053,3053,1,0.0095
407,407,1,0.0095
3045,3045,1,0.0113
1550,1550,1,0.0114
2157,2157,1,0.0129
1218,1218,1,0.0152
51,51,1,0.0174
2435,2435,1,0.0202


In [34]:
fp_examples = false_positives.head(5)[["index", "y_true", "y_prob"]].merge(
    X[["Patient_profile"]],
    left_on="index",
    right_index=True
)

fn_examples = false_negatives.head(5)[["index", "y_true", "y_prob"]].merge(
    X[["Patient_profile"]],
    left_on="index",
    right_index=True
)

In [38]:
fp_examples['Patient_profile'].to_list()

['Age: 37.0\n    Gender: female\n    Life Style: Data not available\n    Family History: older sister died as a neonate shortly after birth from complex congenital heart disease, although no further detail was available\n    Social History: Data not available\n    Medical/Surgical History: Data not available\n    Signs and Symptoms: mild to moderate mitral valve prolapse, accidental discovery of an echogenic mass associated with the aortic root and right atrium; slight shortness of breath complying with nyha grade i, congenital bilateral ptosis, de quervain’s thyroiditis, congenital bilateral hearing problems; cardiac manifestations, saddle nose deformity and camptodactyly; strikingly erythematous rash affecting her thighs, lower legs, flexures of forearms and sparing the knee flexures and popliteal fossa, marked contractures affecting the fingers in both her hands; mild systolic murmur audible over the apex; no diastolic murmur; no evidence of heart failure; cardiomegaly with a slight

In [39]:
fn_examples['Patient_profile'].to_list()

['Age: 28.0\n    Gender: male\n    Life Style: Data not available\n    Family History: only child of a healthy mother; pregnancy started 6 months after termination of chemotherapy of the father’s leukemia (acute lymphoblastic leukemia). the father died at the age of 30 years before birth of his son. the father has 3 healthy sibs, who have no descendants. miscarriages or stillbirths are unknown in the father’s anamnesis. the patient was delivered spontaneously having normal measures\n    Social History: has attended special assisted schooling; is able to read and write simple texts at a basic level and manage his practical life under his mother’s guidance, but would not be able to lead an independent life; has a full time job together with other mentally handicapped people in a ‘‘sheltered’’ supermarket\n    Medical/Surgical History: Data not available\n    Signs and Symptoms: face has a coarse appearance with full eye-brows and full lips, especially the lower lip which is also averted;